# Prever a contagem de anéis em abalones

Caderno do capítulo 14. A resposta é a contagem Rings; os três preditores são comprimento, diâmetro e massa do exemplar inteiro na escala publicada. O teste é aberto na etapa 4. Depois da preparação, pode ir diretamente à etapa 5 para praticar a ligação de ficheiros.

Execute as células pela ordem apresentada. No Colab, use o ZIP completo dos materiais de apoio; no computador, mantenha este caderno na pasta principal. Guarde as suas interpretações em células de texto. As saídas são calculadas quando executar o caderno.


## Preparar os ficheiros


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

# No computador, abra este caderno junto de reproduzir.py.
# No Colab, a primeira execução pede o ZIP completo dos materiais.
candidatos = [Path.cwd(), Path.cwd().parent]
pasta_colab = Path('/content/materiais_biociencias')
if pasta_colab.exists():
    candidatos += [p.parent for p in pasta_colab.rglob('reproduzir.py')]
raiz = next((p for p in candidatos if (p / 'reproduzir.py').is_file()), None)
if raiz is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            'Abra o caderno na pasta que contém reproduzir.py, dados e reproducao.'
        ) from erro
    carregados = files.upload()
    arquivos = [Path(nome) for nome in carregados if nome.lower().endswith('.zip')]
    if len(arquivos) != 1:
        raise ValueError('Carregue um único ZIP completo dos materiais de apoio.')
    pasta_colab.mkdir(parents=True, exist_ok=True)
    destino = pasta_colab.resolve()
    with zipfile.ZipFile(arquivos[0]) as arquivo:
        for item in arquivo.infolist():
            caminho = (destino / item.filename).resolve()
            if not caminho.is_relative_to(destino):
                raise ValueError('O ZIP contém um caminho fora da pasta de destino.')
        arquivo.extractall(destino)
    encontrados = list(destino.rglob('reproduzir.py'))
    if len(encontrados) != 1:
        raise ValueError('O ZIP deve conter uma única pasta completa dos materiais.')
    raiz = encontrados[0].parent
os.chdir(raiz)
print('Pasta de trabalho:', raiz.name)

def executar_python(script, *argumentos):
    subprocess.run([sys.executable, script, *argumentos], check=True, cwd=raiz)


## Conferir o ambiente

Esta célula instala as dependências apenas se necessário. Se pedir um reinício, reinicie o ambiente e volte à primeira célula.


In [ ]:
from importlib.metadata import version, PackageNotFoundError

requisitos = {}
for linha in Path('requisitos.txt').read_text().splitlines():
    if '==' in linha and not linha.lstrip().startswith('#'):
        nome, esperada = linha.strip().split('==', 1)
        requisitos[nome] = esperada
alterar = False
for nome, esperada in requisitos.items():
    try:
        atual = version(nome)
    except PackageNotFoundError:
        atual = 'não instalada'
    print(f'{nome}: {atual} (referência: {esperada})')
    alterar = alterar or atual != esperada
if alterar:
    ja_importadas = any(n in sys.modules for n in ['numpy', 'pandas', 'sklearn', 'matplotlib', 'scipy'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requisitos.txt'], check=True)
    if ja_importadas:
        raise RuntimeError(
            'Dependências instaladas. Reinicie o ambiente e execute desde a primeira célula '
            'para usar as versões agora instaladas.'
        )
else:
    print('O ambiente já tem as versões de referência.')


## 1. Abrir apenas o desenvolvimento

Usamos comprimento, diâmetro e massa do exemplar inteiro na escala publicada. A resposta é Rings: contagem de anéis. O conjunto público Abalone é de Nash et al. (1994), UCI, DOI https://doi.org/10.24432/C55C7W, CC BY 4.0. A documentação informa que os valores contínuos foram divididos por 200. Não interpretar diretamente os valores como mm/g.

In [ ]:
import pandas as pd

dados = pd.read_csv(
    "dados/abalone_desenvolvimento.csv", float_precision="round_trip"
)
entradas = ["Length", "Diameter", "Whole_weight"]
X = dados[entradas]
y = dados["Rings"]

print(dados.shape)
print(dados[entradas + ["Rings"]].describe())
print(dados[entradas + ["Rings"]].isna().sum())


### Figuras da execução de referência

A figura seguinte utiliza apenas o desenvolvimento.

In [ ]:
from IPython.display import display, Image
display(Image(filename="figuras/u14_abalone_desenvolvimento.png"))


## 2. Comparar os procedimentos

Os três procedimentos usam as mesmas cinco partes. Escolhemos o menor MAE médio antes do arredondamento. A diferença entre regressão linear e floresta é muito pequena e não demonstra superioridade prática ou estatística.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

particoes = list(KFold(
    n_splits=5, shuffle=True, random_state=2026
).split(X, y))

candidatos = {
    "Referencia": DummyRegressor(strategy="median"),
    "Regressao_linear": LinearRegression(),
    "Floresta": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5,
        max_features=1.0, random_state=2026, n_jobs=1
    )
}

linhas = []
for nome, procedimento in candidatos.items():
    resultado = cross_validate(
        procedimento, X, y, cv=particoes,
        scoring={"mae": "neg_mean_absolute_error",
                 "rmse": "neg_root_mean_squared_error"}
    )
    mae = -resultado["test_mae"]
    rmse = -resultado["test_rmse"]
    linhas.append({
        "Procedimento": nome,
        "MAE_medio": mae.mean(),
        "Desvio_MAE": mae.std(ddof=1),
        "RMSE_medio": rmse.mean()
    })

resumo = pd.DataFrame(linhas)
escolhido = resumo.sort_values(
    "MAE_medio", kind="stable"
).iloc[0]["Procedimento"]
print(resumo)
print("Procedimento escolhido:", escolhido)


## 3. Guardar a decisão

Este registo precede a abertura do teste. As configurações completas estão no código acima e em reproducao/abalone/protocolo.json. A reprodução depois de conhecer os resultados não constitui uma nova avaliação independente desse conhecimento.

In [ ]:
from pathlib import Path
import json

saida = Path("resultados_abalone")
saida.mkdir(exist_ok=True)
decisao = {
    "procedimento": escolhido,
    "entradas": entradas,
    "resposta": "Rings",
    "criterio": "menor MAE medio na validacao",
    "previsoes": "sem arredondamento nem limitacao"
}
(saida / "procedimento_escolhido.json").write_text(
    json.dumps(decisao, ensure_ascii=False, indent=2),
    encoding="utf-8"
)


## 4. Avaliar o procedimento escolhido

Só depois de guardar a decisão abrimos o teste, reajustamos o escolhido e a referência no desenvolvimento e calculamos os erros. Conservamos previsões contínuas, sem arredondamento. O MAE é uma média em número de anéis e não um limite individual.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score
)

teste = pd.read_csv(
    "dados/abalone_teste_reservado.csv", float_precision="round_trip"
)
X_teste = teste[entradas]
y_teste = teste["Rings"]

modelo_final = clone(candidatos[escolhido]).fit(X, y)
referencia_final = clone(candidatos["Referencia"]).fit(X, y)
previsao = modelo_final.predict(X_teste)
previsao_referencia = referencia_final.predict(X_teste)

for nome, valores in [("Referencia", previsao_referencia),
                     (escolhido, previsao)]:
    print(nome)
    print("MAE:", mean_absolute_error(y_teste, valores))
    print("RMSE:", root_mean_squared_error(y_teste, valores))
    print("R2:", r2_score(y_teste, valores))


### Interpretar o teste de referência

A figura mostra as previsões da execução documentada. Observe a subestimação das contagens mais elevadas. As faixas diagonais nos resíduos resultam também da natureza inteira da resposta; por si só, não demonstram falha do ajustamento.

In [ ]:
display(Image(filename="figuras/u14_abalone_teste.png"))


## 5. Reconstituir a avaliação por identificador

Este bloco pode ser executado diretamente com os ficheiros de resultados incluídos, sem executar os ajustamentos. Confirma o conjunto de IDs previsto, deteta duplicações/ausências, junta as previsões e as respostas e identifica os maiores erros. Reconstitui a mesma avaliação; não é um novo teste independente.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pasta = Path("reproducao/abalone/resultados")
referencia = pd.read_csv(
    pasta / "referencia_avaliacao.csv",
    dtype={"id_registo": "string"}, float_precision="round_trip"
)
recebidas = pd.read_csv(
    pasta / "previsoes_avaliacao.csv",
    dtype={"id_registo": "string"}, float_precision="round_trip"
)

plano = pd.read_csv(
    "dados/abalone_teste_reservado.csv", usecols=["id_registo"],
    dtype={"id_registo": "string"}
)
ids_previstos = set(plano["id_registo"])
if not ids_previstos or plano["id_registo"].isna().any():
    raise ValueError("O plano de avaliacao nao e valido.")
if not plano["id_registo"].is_unique:
    raise ValueError("O plano contem identificadores repetidos.")

for tabela in [referencia, recebidas]:
    if tabela["id_registo"].isna().any():
        raise ValueError("Ha identificadores em falta.")
    if set(tabela["id_registo"]) != ids_previstos:
        raise ValueError("O ficheiro nao corresponde ao plano.")

avaliacao = referencia.merge(
    recebidas, on="id_registo", how="outer",
    validate="one_to_one", indicator=True
)
if not avaliacao["_merge"].eq("both").all():
    raise ValueError("Os identificadores nao correspondem.")

colunas = ["Rings", "previsao"]
valores = avaliacao[colunas].to_numpy(dtype=float)
if not np.isfinite(valores).all():
    raise ValueError("Ha respostas ou previsoes nao finitas.")

avaliacao["residuo"] = (
    avaliacao["Rings"] - avaliacao["previsao"]
)
avaliacao["erro_absoluto"] = avaliacao["residuo"].abs()
print(avaliacao["erro_absoluto"].mean())
print(avaliacao.nlargest(5, "erro_absoluto"))


## Concluir

Escreva uma conclusão com a pergunta, a seleção em validação, o MAE no teste, a comparação com a referência e uma limitação. Distinga a contagem de anéis de uma idade cronológica verificada. A avaliação não demonstra desempenho numa região ou época nova.

A reprodução completa, incluindo a preparação e a geração das figuras, está em `reproducao/abalone/codigo/reproduzir_abalone.py`.